# Teaching Your Agent to Remember: Working with Memories

In this notebook, you will learn how to give your AI agent a **memory**. By default, agents forget everything after each conversation. With memory stores, your agent can remember facts about users — like their name, preferences, and past conversations — so it can respond in a more personalized way.

We will walk through every step, from installing packages to chatting with an agent that actually remembers you!

**Prerequisites — Two model deployments are required in your Foundry project:**
1. A chat model (e.g., `gpt-4.1-mini`) — powers the agent's reasoning
2. An embedding model: **`text-embedding-3-small`** — converts text into searchable vectors for memory lookup. Deploy this from the Foundry portal under Models + endpoints before running this notebook.

## Step 1: Install Packages

Before we write any code, we need to install a few Python libraries. These give us the tools to connect to Azure AI Foundry and work with AI models.

In [ ]:
# Install the required libraries:
# - azure-ai-projects: lets us interact with Azure AI Foundry (agents, memory, etc.)
# - openai: the OpenAI SDK used for chat completions
# - python-dotenv: loads secret settings from a .env file so we don't hardcode them
# - azure-identity: handles authentication with Azure services
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

## Step 2: Load Settings

We store sensitive information (like endpoints and model names) in a `.env` file. This keeps our secrets out of the code. The `load_dotenv()` function reads that file and makes those values available through `os.getenv()`.

Our `.env` file contains three settings:
- `FOUNDRY_PROJECT_ENDPOINT` — the URL of our Foundry project
- `MODEL_DEPLOYMENT_NAME` — the chat model (e.g., `gpt-4.1-mini`)
- `TEXT_EMBEDDING_MODEL_NAME` — must be set to **`text-embedding-3-small`** (required for memory vector search)

In [ ]:
# Import the libraries we need for this step
import os                                      # Lets us read environment variables
from dotenv import load_dotenv                  # Reads the .env file into memory
from azure.identity import DefaultAzureCredential  # Handles Azure login automatically
from azure.ai.projects import AIProjectClient  # The main client for Azure AI Foundry

# Load all the key-value pairs from the .env file into environment variables
load_dotenv()

# Read each setting from the environment
# my_endpoint: the URL of our Azure AI Foundry project
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

# my_model: the name of the AI model we want the agent to use (e.g., GPT-4)
my_model = os.getenv("MODEL_DEPLOYMENT_NAME")

# embedding_model: a model that turns text into numbers (vectors) for memory search
embedding_model = os.getenv("TEXT_EMBEDDING_MODEL_NAME")

# Print the embedding model name to verify it loaded correctly
print("Embedding model loaded:", embedding_model)

## Step 3: Connect to Foundry

Now we create a **client** — think of it as a phone line to Azure AI Foundry. Every time we want to do something (create an agent, store memories, etc.), we use this client to make the call.

In [ ]:
# Create the Foundry client by providing:
#   - endpoint: where the service lives on the internet
#   - credential: proof that we are allowed to use it (Azure handles this automatically)
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=DefaultAzureCredential()
)

## Step 4: Create a Memory Store

A **memory store** is like a notebook where the agent writes down things it learns about users. It uses:
- A **chat model** (`gpt-4.1-mini`) to understand and summarize conversations
- An **embedding model** (`text-embedding-3-small`) to convert text into searchable vectors — this is what makes memory retrieval fast and accurate

We also enable two handy features:
- `user_profile_enabled`: the agent builds a profile of each user over time
- `chat_summary_enabled`: the agent keeps summaries of past conversations

In [ ]:
# Import the classes we need to define and configure a memory store
from azure.ai.projects.models import MemoryStoreDefaultDefinition, MemoryStoreDefaultOptions

# The unique name for our memory store
memory_store_name = "demo_memory_store_v1"

# Set up the memory store configuration
# This tells Azure which models to use and which features to turn on
memory_config = MemoryStoreDefaultDefinition(
    chat_model=my_model,              # The AI model that processes memory content
    embedding_model=embedding_model,   # The model that creates searchable vectors
    options=MemoryStoreDefaultOptions(
        user_profile_enabled=True,     # Build a profile for each user automatically
        chat_summary_enabled=True      # Keep summaries of past chats
    )
)

# Create the memory store, or retrieve it if it already exists from a previous run
try:
    my_memory_store = foundry_client.memory_stores.create(
        name=memory_store_name,
        definition=memory_config,
        description="A memory store to retain user context across sessions"
    )
    print("Memory store created! Name:", my_memory_store.name)
except Exception as e:
    if "already exists" in str(e):
        # The store was created in a previous run — just retrieve it
        my_memory_store = foundry_client.memory_stores.get(name=memory_store_name)
        print("Memory store already exists — reusing it! Name:", my_memory_store.name)
    else:
        raise  # Re-raise if it's a different error

## Step 5: Build a Helper Function to Save Memories

We will create a reusable function called `save_to_memory`. Whenever we want the agent to remember something about a user, we call this function with:
- The name of the memory store
- The user's name (used as a "scope" to keep each user's memories separate)
- The text we want to save

In [ ]:
# Import the message type that represents a user's message
from azure.ai.projects.models import ResponsesUserMessageItemParam

def save_to_memory(store_name: str, user_id: str, text_to_remember: str):
    """
    Saves a piece of information to the memory store for a specific user.
    
    Parameters:
        store_name (str):       The name of the memory store to save to
        user_id (str):          The user's name — used to keep memories organized per person
        text_to_remember (str): The actual text content to store as a memory
    
    Returns:
        str: "updated successfully" or "update failed"
    """
    try:
        # The scope links this memory to a specific user
        scope = user_id

        # Wrap the text in a message object that the API expects
        message_item = ResponsesUserMessageItemParam(
            content=text_to_remember
        )

        # Send the memory to Azure and wait for it to finish processing
        # update_delay=0 means "process this immediately, don't wait"
        save_operation = foundry_client.memory_stores.begin_update_memories(
            name=store_name,
            scope=scope,
            items=[message_item],
            update_delay=0
        )

        # Wait until the save is complete and get the result
        result = save_operation.result()

        # Print out what happened so we can see the memories that were created
        print(f"Saved {len(result.memory_operations)} memory operation(s):")
        for op in result.memory_operations:
            print(f"  - Type: {op.kind}, ID: {op.memory_item.memory_id}, Content: {op.memory_item.content}")

        return "updated successfully"

    except Exception as err:
        # If something goes wrong, print the error and return a failure message
        print("Something went wrong while saving memory:", str(err))
        return "update failed"

## Step 6: Store Some Initial Facts

Let's give the agent something to remember about our demo user, **Alex**. We will save a short bio that includes personal preferences. Later, the agent will recall these facts during conversation.

In [ ]:
# Save some facts about our demo user "alex" into the memory store
save_to_memory(
    store_name=my_memory_store.name,
    user_id="alex",
    text_to_remember="My name is Alex and I enjoy hiking and photography. I'm a vegetarian and prefer morning workouts."
)

## Step 7: Create an Agent

Now we create the AI agent that will serve as our personalized assistant. The agent itself doesn't have the memory tool attached directly — instead, we will **manually search** the memory store in Step 8 and pass any relevant memories into each request as extra context.

> **Note:** Azure AI Foundry offers a `MemorySearchTool` that lets the agent search memories automatically. However, this feature is only available in certain Azure regions. If your project is in a supported region, you can attach the tool directly to the agent (see the [Azure docs](https://learn.microsoft.com/azure/ai-foundry/concepts/memory) for details). In this notebook, we use the manual approach so it works everywhere.

In [ ]:
# Import the class we need for the agent definition
from azure.ai.projects.models import PromptAgentDefinition

# Choose a name for our agent — this is how we reference it later
remembering_agent_name = "memory-assistant"

# Create the agent on Azure AI Foundry
# We don't attach the memory tool here — instead we manually search
# the memory store and pass context in Step 8
remembering_agent = foundry_client.agents.create_version(
    agent_name=remembering_agent_name,
    definition=PromptAgentDefinition(
        model=my_model,                        # Which AI model powers this agent
        instructions=(
            "You are a personalized assistant that remembers user preferences "
            "and past conversations to provide tailored responses. "
            "When memory context is provided, use it to personalize your answers."
        ),
    ),
)

# Print confirmation with the agent's details
print(f"Agent created! ID: {remembering_agent.id}, Name: {remembering_agent.name}, Version: {remembering_agent.version}")

## Step 8: Chat with Memory-Powered Responses

This is where it all comes together! The chat loop below does the following on every message:

1. **Takes your question** as input
2. **Searches the memory store** for any facts related to your question
3. **Appends those memories** to the question so the agent has context
4. **Sends the enriched question** to the agent and prints the personalized response

You can also type `update` to add new facts to the memory store at any time.

In [ ]:
# Import the memory search options class
from azure.ai.projects.models import MemorySearchOptions

# This flag controls whether the chat loop keeps running
keep_chatting = True

# Get an OpenAI-compatible client from our Foundry client (created once, reused each loop)
oai_client = foundry_client.get_openai_client()

# Ask the user to identify themselves (or type 'exit' to skip)
active_user = input("What is your name? (type 'exit' to quit): ")

# Main chat loop — runs until the user types 'exit'
while keep_chatting:

    # Get the user's message
    user_input = input('Type "exit" to quit, "update" to save new info to memory, or ask a question: ')

    # Check if the user wants to leave
    if user_input.lower() == "exit":
        keep_chatting = False

    # Check if the user wants to add a new memory
    elif user_input.strip().upper() == "UPDATE":
        new_fact = input("Enter the information you want to save to memory: ")
        save_to_memory(
            store_name=my_memory_store.name,
            user_id=active_user,
            text_to_remember=new_fact
        )

    else:
        # --- Normal question flow ---

        # Wrap the user's question in the format the memory search expects
        search_query = ResponsesUserMessageItemParam(content=user_input)

        # Search the memory store for facts related to the user's question
        # max_memories=5 means "return at most 5 relevant memories"
        found_memories = foundry_client.memory_stores.search_memories(
            name=my_memory_store.name,
            scope=active_user,
            items=[search_query],
            options=MemorySearchOptions(max_memories=5)
        )

        # Build the enriched prompt: start with the user's original question
        enriched_input = user_input

        # If we found any memories, append them as extra context for the agent
        if len(found_memories.memories) > 0:
            for mem in found_memories.memories:
                print(f"  [Memory found]: {mem.memory_item.content}")
                enriched_input += f"\nMemory Context: {mem.memory_item.content}"

        # Send the enriched question to the agent and get a response
        agent_reply = oai_client.responses.create(
            # Tell the service which agent should handle this request
            extra_body={
                "agent": {
                    "name": remembering_agent_name,
                    "type": "agent_reference"
                }
            },
            # The user's question, enriched with any matching memories
            input=enriched_input
        )

        # Print the agent's response
        print(f"Agent: {agent_reply.output_text}\n")